# Candidate Generation: Rolling Popularity Baseline

Non-personalised control baseline that ranks each request by **recent article activity** instead of split-wide popularity. It is the missing comparator for the content two-tower: the two-tower's dominant feature is the previous completed hour of exposures/clicks, so a candidate generator that uses the same signal without any user model is required to interpret its Recall@M.

Protocol:

- Build hourly exposure and click counts per article from the logged candidate slates (`candidates_<split>.parquet`).
- For a request at hour `h`, use only hours `< h` (prequential, no leakage).
- Rank articles by previous-hour clicks, tie-broken by previous-hour exposures and then ascending `news_code`.
- Restrict candidates to `available_from <= request timestamp`.
- Write the standard `impression_uid, cutoff, news_code` table and evaluate Recall@M with `cg_evaluation.py`.

The score mode is fixed to `prior_hour_clicks`: clicks in the previous completed hour, tie-broken by exposure count. It is not configurable.

## 1. Setup

In [1]:
from pathlib import Path
import os
import sys

import numpy as np
import polars as pl

CODE_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'run_all.py').is_file()
)
ROOT = Path(os.environ.get('NEWSREC_ROOT', CODE_ROOT)).expanduser().resolve()
sys.path.insert(0, str(CODE_ROOT / '5_Models' / '5_01_Candidate_Generation'))

from cg_evaluation import CandidateEvaluator
from cg_result_writer import CandidateWriter

PROCESSED = ROOT / '1_Data_Preparation' / 'processed'
REPRESENTATIONS = ROOT / '3_Representations' / 'artifacts'
ARTIFACTS = ROOT / '5_Models' / '5_01_Candidate_Generation' / 'artifacts'
RESULTS = ROOT / 'results'

TARGET_SPLIT = 'validation'
M_VALUES = (50, 100, 200, 250, 300, 350, 400, 450, 500)
CANDIDATE_CAP = max(M_VALUES)

HOUR_NS = 3_600 * 10**9
POOL_MARGIN = 4_000
WRITE_BATCH = 4_096

if TARGET_SPLIT not in {'validation', 'test'}:
    raise ValueError('TARGET_SPLIT must be validation or test')

MODEL_NAME = 'popularity_freshness_prior_hour_clicks'
OUTPUT_DIR = ARTIFACTS / MODEL_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)

SOURCE_SPLITS = (
    ['train', 'validation'] if TARGET_SPLIT == 'validation'
    else ['train', 'validation', 'test']
)

print('Repository:', ROOT)
print('Target split:', TARGET_SPLIT)
print('Source splits:', SOURCE_SPLITS)
print('Score mode: prior_hour_clicks')
print('Candidate cap:', CANDIDATE_CAP)

Repository: C:\Users\Yong Rui\BT4222-Group-6-
Target split: validation
Source splits: ['train', 'validation']
Score mode: prior_hour_clicks
Candidate cap: 500


## 2. Catalogue, availability and target impressions

Article rows match the shared `article_index` convention (`article_row = 1..N`). The padding row `0` is never eligible.

In [2]:
# Article catalogue: article_row (1..N) -> news_code, with each article's observable available_from.
meta = (
    pl.read_parquet(
        REPRESENTATIONS / 'article_index.parquet',
        columns=['article_row', 'news_code'],
    )
    .sort('article_row')
    .join(
        pl.read_parquet(
            REPRESENTATIONS / 'article_availability.parquet',
            columns=['news_code', 'available_from'],
        ),
        on='news_code',
        how='left',
    )
)

# n_items counts the padding row 0, so real articles are rows 1..N.
n_items = meta.height + 1

# Lookup arrays to translate between article_row and news_code in both directions.
rows = meta['article_row'].to_numpy().astype(np.int64)
codes = meta['news_code'].to_numpy().astype(np.int64)
row_of_code = np.zeros(int(codes.max()) + 1, dtype=np.int64)
row_of_code[codes] = rows
row_to_code = np.zeros(n_items, dtype=np.int32)
row_to_code[rows] = codes.astype(np.int32)
# available_from per article_row in epoch ns; padding row 0 stays at i64 max so it is never eligible.
avail_row = np.full(n_items, np.iinfo(np.int64).max, dtype=np.int64)
avail_row[rows] = meta['available_from'].dt.epoch('ns').to_numpy().astype(np.int64)

# Target split impressions: the requests we must generate candidates for.
impressions = pl.read_parquet(
    PROCESSED / f'impressions_{TARGET_SPLIT}.parquet',
    columns=['impression_uid', 'timestamp'],
)

print(f'Catalogue articles: {n_items - 1:,}')
print(f'Target impressions: {impressions.height:,}')

Catalogue articles: 51,282
Target impressions: 31,624


## 3. Build the hourly engagement state

Aggregate the logged candidate slates by `(news_code, hour)` for all source splits into dense `(hour x article)` exposure and click matrices. Nothing is shifted or accumulated here: each row `h` holds only the counts logged during clock hour `h`, and the scorer later reads row `h - 1` so every feature is strictly causal.

In [3]:
# Reference point for the hour index: midnight of the first train day.
first_ts = (
    pl.scan_parquet(PROCESSED / 'candidates_train.parquet')
    .select(pl.col('timestamp').min())
    .collect()
    .item()
)
T0_NS = int(
    np.datetime64(first_ts, 'ns').astype('datetime64[D]').astype('datetime64[ns]').astype(np.int64)
)

# Aggregate every source split into (news_code, hour) exposure and click counts.
hourly = (
    pl.concat([
        pl.scan_parquet(PROCESSED / f'candidates_{split}.parquet')
        .select(
            'news_code',
            'clicked',
            ((pl.col('timestamp').dt.epoch('ns') - T0_NS) // HOUR_NS).alias('hour'),
        )
        for split in SOURCE_SPLITS
    ])
    .group_by('news_code', 'hour')
    .agg(
        exp=pl.len(),
        clk=pl.col('clicked').cast(pl.Int64).sum(),
    )
    .collect()
)

# Map news_code to article_row for the dense matrices below.
source_rows = row_of_code[hourly['news_code'].to_numpy().astype(np.int64)]

# Dense (hour x article) matrices so a snapshot for hour h is just row h-1.
N_HOURS = int(hourly['hour'].max()) + 1
EXPOSURES = np.zeros((N_HOURS, n_items), dtype=np.int32)
CLICKS = np.zeros((N_HOURS, n_items), dtype=np.int32)
hour_idx = hourly['hour'].to_numpy()
EXPOSURES[hour_idx, source_rows] = hourly['exp'].to_numpy()
CLICKS[hour_idx, source_rows] = hourly['clk'].to_numpy()

print(f'Source candidate rows: {int(EXPOSURES.sum()):,}')
print(f'Hourly states: {N_HOURS:,}')
print(f'Articles exposed at least once: {int((EXPOSURES.sum(0) > 0).sum()):,}')

Source candidate rows: 4,621,015
Hourly states: 120
Articles exposed at least once: 16,978


## 4. Non-personalised hourly scorer

`score_snapshot(h)` uses only hours `< h` and always scores by prior-hour clicks. `eligible_order(h)` restricts to articles available by the end of hour `h`, then orders by prior-hour clicks descending, prior-hour exposures descending, and `news_code` ascending. The exact request timestamp is applied during candidate selection, so articles that appear later in the same hour are still handled correctly.

In [4]:
def score_snapshot(hour):
    """Return (prior-hour clicks, prior-hour exposures)."""
    if hour == 0:
        zero = np.zeros(n_items, dtype=np.int32)
        return zero, zero
    return CLICKS[hour - 1], EXPOSURES[hour - 1]


def eligible_order(hour):
    hour_end = T0_NS + (hour + 1) * HOUR_NS
    eligible = np.flatnonzero(avail_row <= hour_end)
    score, tiebreak = score_snapshot(hour)
    return eligible[
        np.lexsort(
            (row_to_code[eligible], -tiebreak[eligible], -score[eligible])
        )
    ]

## 5. Generate the standard candidate table

Requests are grouped by hour. One score order is built per hour, then each request filters that order by its exact `available_from <= timestamp` condition. The pool grows adaptively if mid-hour availability removes too many candidates.

In [5]:
target = (
    impressions
    .with_columns(
        ((pl.col('timestamp').dt.epoch('ns') - T0_NS) // HOUR_NS).alias('hour')
    )
    .sort(['hour', 'timestamp', 'impression_uid'])
)
uid = target['impression_uid'].to_numpy().astype(np.int64)
ts = target['timestamp'].dt.epoch('ns').to_numpy().astype(np.int64)
hour = target['hour'].to_numpy().astype(np.int64)
if hour.min() < 0 or hour.max() >= N_HOURS:
    raise ValueError('Target impressions fall outside the hourly state range')

recommendations_path = (
    OUTPUT_DIR / f'recommendations_{TARGET_SPLIT}_top{CANDIDATE_CAP}.parquet'
)
cutoff_values = np.arange(1, CANDIDATE_CAP + 1, dtype=np.int16)

# 'hour' is already sorted, so contiguous runs give each hour's request slice.
starts = np.flatnonzero(np.r_[True, hour[1:] != hour[:-1]])
ends = np.r_[starts[1:], len(hour)]
unique_hours = hour[starts]

writer = CandidateWriter(recommendations_path)
try:
    for h, start, end in zip(unique_hours.tolist(), starts.tolist(), ends.tolist()):
        ranked = eligible_order(h)
        base_pool = min(ranked.size, CANDIDATE_CAP + POOL_MARGIN)
        # Doubling ladder of pool sizes, capped at the full ranking.
        pools = sorted({min(ranked.size, base_pool << k) for k in range(ranked.size.bit_length())})
        hour_uid = uid[start:end]
        hour_ts = ts[start:end]
        for b0 in range(0, hour_uid.size, WRITE_BATCH):
            batch_uid = hour_uid[b0:b0 + WRITE_BATCH]
            batch_ts = hour_ts[b0:b0 + WRITE_BATCH]
            selected = np.empty((batch_ts.size, CANDIDATE_CAP), dtype=np.int64)
            for j, request_ts in enumerate(batch_ts.tolist()):
                for pool in pools:
                    candidate_rows = ranked[:pool]
                    chosen = candidate_rows[avail_row[candidate_rows] <= request_ts]
                    if chosen.size >= CANDIDATE_CAP:
                        selected[j] = chosen[:CANDIDATE_CAP]
                        break
            writer.add(
                np.repeat(batch_uid, CANDIDATE_CAP),
                np.tile(cutoff_values, batch_uid.size),
                row_to_code[selected].reshape(-1),
            )
finally:
    writer.close()

expected_rows = len(uid) * CANDIDATE_CAP
saved_rows = pl.scan_parquet(recommendations_path).select(pl.len()).collect().item()
if saved_rows != expected_rows:
    raise ValueError(f'Expected {expected_rows:,} rows, found {saved_rows:,}')

print(f'Unique hours: {len(unique_hours):,}')
print(f'Requests scored: {len(uid):,}')
print(f'Candidate rows: {saved_rows:,}')
print('Candidates saved to:', recommendations_path)

Unique hours: 24
Requests scored: 31,624
Candidate rows: 15,812,000
Candidates saved to: C:\Users\Yong Rui\BT4222-Group-6-\5_Models\5_01_Candidate_Generation\artifacts\popularity_freshness_prior_hour_clicks\recommendations_validation_top500.parquet


## 6. Evaluation

Uses the shared `CandidateEvaluator` and writes `results/candidate_generation_<model>_<split>.csv`.

In [6]:
evaluator = CandidateEvaluator(root=ROOT, split=TARGET_SPLIT)
evaluation = evaluator.evaluate_to_csv(
    recommendations_path,
    model_name=MODEL_NAME,
    m_values=M_VALUES,
    output_dir=RESULTS,
)
evaluation_path = RESULTS / f'candidate_generation_{MODEL_NAME}_{TARGET_SPLIT}.csv'
print('Comparison updated:', RESULTS / f'candidate_generation_comparison_{TARGET_SPLIT}.csv')
print('Evaluation saved to:', evaluation_path)
print(evaluation.sort('m'))

Comparison updated: C:\Users\Yong Rui\BT4222-Group-6-\results\candidate_generation_comparison_validation.csv
Evaluation saved to: C:\Users\Yong Rui\BT4222-Group-6-\results\candidate_generation_popularity_freshness_prior_hour_clicks_validation.csv
shape: (72, 7)
┌───────────────────┬────────────┬─────┬───────────────────┬───────────┬─────────────┬─────────────┐
│ model             ┆ split      ┆ m   ┆ metric            ┆ value     ┆ numerator   ┆ denominator │
│ ---               ┆ ---        ┆ --- ┆ ---               ┆ ---       ┆ ---         ┆ ---         │
│ str               ┆ str        ┆ i64 ┆ str               ┆ f64       ┆ f64         ┆ f64         │
╞═══════════════════╪════════════╪═════╪═══════════════════╪═══════════╪═════════════╪═════════════╡
│ popularity_freshn ┆ validation ┆ 50  ┆ catalogue_coverag ┆ 0.006879  ┆ 327.0       ┆ 47538.0     │
│ ess_prior_hou…    ┆            ┆     ┆ e                 ┆           ┆             ┆             │
│ popularity_freshn ┆ validatio

## Notes and limitations

- It is non-personalised: every request in the same hour receives the same base order, apart from availability.
- `prior_hour_clicks` ranks by realised clicks in the previous completed hour, so it is less tied to the logging system's slate than the exposure control, but it is still confounded by position and by which articles were shown. Click counts are sparser than exposures (roughly 0.5-1.5k articles clicked per hour versus ~3k exposed), so the exposure tie-break matters mainly in low-traffic hours.
- Article history is not excluded. Only 0.05% of validation clicks are on articles already in the user's `full_history_rows`, so the difference from the neural models' history-exclusion policy is negligible for Recall@M.
- The rolling features are built from the logged candidate slates, which include the realised click labels. Only hours strictly before the request hour are used, so no target-period information enters the score.
- `available_from` is the project-wide observable-availability proxy, not a verified publication timestamp.
- Use this model as the control when claiming that a personalised or neural generator improves candidate recall.